# Phase 40 — clean full-rate 2-worker run (lockstep, single GPU)

The T4×2 run desynced (two GPUs at different speeds → fixed-iteration workers exited at
different rounds; one ran the tail solo). This run removes that confound with a **lockstep**
driver: ONE process, ONE θ, **two worker-ids** submitting one proposal each per round, so the
server (`TARGET_PROPOSALS=2`) advances exactly one round per iteration — deterministic
full-rate co-training to exactly **R=300**, no desync, no two-GPU float nondeterminism, a
bit-identical replica by construction. The cross-audit + η-adaptation paths are exercised
identically (two distinct worker-ids, no-self-audit).

**Honest framing:** this measures the learning curve under full-rate + η-adaptation cleanly;
it does not claim two *physical* machines. Win condition: a monotone held-out drop, η climbing,
and a final number we can trust (vs the retracted drift-inflated −0.0089).

## Before ▶ Run All
1. **Accelerator** → any single `GPU` (T4 is fine; x2 not needed). 2. **Internet** → On.

~20–30 min. Resets the deployed coordinator first. Outputs in `/kaggle/working/learn3/`.


## Cell 1 — install deps + clone repos (pulls the lockstep verifier)

In [ ]:
import subprocess, os, sys, urllib.request
try:
    urllib.request.urlopen("https://github.com", timeout=5).close(); print("✓ internet")
except Exception as e:
    sys.exit(f"✗ INTERNET DISABLED: {e}\n  Right sidebar (gear) → Internet → On, then re-run.")
print("→ pip install …")
subprocess.run(["pip", "-q", "install", "transformers", "torch", "numpy", "requests", "matplotlib"], check=True)
os.makedirs("/kaggle/working", exist_ok=True)
for name, url in [("ntkmirror", "https://github.com/leochlon/ntkmirror.git"),
                  ("postnet-cf", "https://github.com/abgnydn/postnet-cf.git")]:
    dst = f"/kaggle/working/{name}"
    if os.path.isdir(dst):
        print(f"→ git pull {name}"); subprocess.run(["git", "-C", dst, "pull", "--ff-only"], check=True)
    else:
        print(f"→ git clone {name}"); subprocess.run(["git", "clone", url, dst], check=True)
print("→ pip install -e ntkmirror")
subprocess.run(["pip", "-q", "install", "-e", "/kaggle/working/ntkmirror"], check=True)
print("OK")

## Cell 2 — build held-out corpus (32 / 32, disjoint; identical to prior runs)

In [ ]:
import json, random, os
os.makedirs("/kaggle/working/learn3", exist_ok=True)

def solve(a, b):
    o = (a % 10) + (b % 10); t = (a // 10) + (b // 10) + (o // 10)
    return (f" Add ones: {a%10}+{b%10}={o}, write {o%10} carry {o//10}. "
            f"Tens: {a//10}+{b//10}+{o//10}={t}. Answer: {a+b}")

rng = random.Random(40); pairs = set()
while len(pairs) < 64:
    a, b = rng.randint(10, 89), rng.randint(10, 89)
    if a + b < 100: pairs.add((a, b))
pairs = list(pairs); rng.shuffle(pairs)
train, evl = pairs[:32], pairs[32:64]
assert not (set(train) & set(evl))
def write(path, ps):
    with open(path, "w") as f:
        for a, b in ps:
            f.write(json.dumps({"prompt": f"Problem: {a} + {b} = ?\nSolution:", "completion": solve(a, b)}) + "\n")
write("/kaggle/working/learn3/train.jsonl", train)
write("/kaggle/working/learn3/eval.jsonl", evl)
print(f"train={len(train)} eval={len(evl)} disjoint ✓")

## Cell 3 — GPU check + reset coordinator

In [ ]:
import torch, requests
assert torch.cuda.is_available(), "No GPU. Right sidebar → Accelerator → GPU."
print("✓ GPU:", torch.cuda.get_device_name(0))
COORD = "https://postnet-cf.abgunaydin94.workers.dev"
_UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
       "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36 postnet-ntk/1.0")
_S = requests.Session(); _S.headers.update({"User-Agent": _UA})
print("reset:", _S.post(f"{COORD}/api/ntk/reset").json())
s = _S.get(f"{COORD}/api/ntk/state").json()
print(f"R={s['round']} eta={s['eta']} K={s.get('K')} target={s.get('target')}")

## Cell 4 — lockstep run: 2 virtual workers, R=300

`--virtual-workers 2` runs both logical workers in one process against one θ; the server
advances one round per iteration. Reaches exactly R=300, no early exit.

In [ ]:
import subprocess, time
log = "/kaggle/working/learn3/run.log"
cmd = [
    "python", "/kaggle/working/postnet-cf/scripts/ntk-verifier.py",
    "--coord", COORD, "--model", "Qwen/Qwen2.5-0.5B-Instruct",
    "--train",    "/kaggle/working/learn3/train.jsonl",
    "--eval",     "/kaggle/working/learn3/eval.jsonl",
    "--artifact", "/kaggle/working/postnet-cf/public/data/qwen05b-math-gates-k5000.bin",
    "--rounds", "300", "--trials", "4",
    "--batch-size", "32", "--eval-batch-size", "32", "--max-length", "64",
    "--device", "cuda", "--dtype", "fp32",
    "--seed", "40", "--worker-id", "kaggle-lockstep",
    "--virtual-workers", "2",
    "--trajectory", "/kaggle/working/learn3/trajectory.csv",
    "--reset",
]
t0 = time.time()
with open(log, "w") as f:
    rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
print(f"exit={rc.returncode}   wall={time.time()-t0:.0f}s")
print("\n".join(open(log).read().splitlines()[-25:]))

## Cell 5 — plot + RESULT3.md (vs single-worker baseline + retracted pre-fix)

In [ ]:
import csv, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
R, T, E, ETA = [], [], [], []
with open("/kaggle/working/learn3/trajectory.csv") as f:
    for row in csv.DictReader(f):
        R.append(int(row["round"])); T.append(float(row["train_loss"]))
        E.append(float(row["eval_loss"]) if row["eval_loss"] else None)
        ETA.append(float(row["eta"]) if row["eta"] else None)

plt.figure(figsize=(8, 5))
plt.plot(R, T, label="train (32 problems)", lw=1.5)
er = [r for r, e in zip(R, E) if e is not None]; ee = [e for e in E if e is not None]
if ee: plt.plot(er, ee, label="held-out eval (32 unseen)", lw=1.5)
plt.xlabel("server round"); plt.ylabel("cross-entropy loss")
plt.title("Lockstep 2-worker gate training — train vs held-out (K=5000, Qwen-0.5B)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("/kaggle/working/learn3/descent3.png", dpi=120); print("saved descent3.png")

def ends(xs):
    xs = [x for x in xs if x is not None]
    return (xs[0], xs[-1], xs[-1]-xs[0]) if xs else (None, None, None)
t0v, t1v, td = ends(T); e0, e1, ed = ends(E)
eta_final = next((x for x in reversed(ETA) if x is not None), None)
def f4(x): return f"{x:.4f}" if x is not None else "—"
def fd(x): return f"{x:+.4f}" if x is not None else "—"
md_txt = f"""# Phase 40 — clean lockstep 2-worker run

- **Mode:** single process, 2 virtual workers in lockstep, one θ, full rate to R={len(R)}.
- **Corpus:** 32 train / 32 held-out (disjoint, identical to all prior runs).
- **η final:** {f4(eta_final)} (init 1e-3).

| metric | start | final | Δ |
|---|---|---|---|
| train loss | {f4(t0v)} | {f4(t1v)} | {fd(td)} |
| held-out eval loss | {f4(e0)} | {f4(e1)} | {fd(ed)} |

**Comparison (held-out Δ):** single-worker baseline −0.0029 (η frozen) · pre-fix 2-worker
~~−0.0089~~ (retracted, drift artifact) · **this run {fd(ed)}** at R={len(R)}, η→{f4(eta_final)},
replica-consistent.

Held-out (never trained on) vs train should track in lockstep with no overfitting gap.

![descent3](descent3.png)
"""
open("/kaggle/working/learn3/RESULT3.md", "w").write(md_txt)
print(md_txt)

## Cell 6 — outputs

In [ ]:
import os
print(os.listdir("/kaggle/working/learn3"))